# 2-4. LCEL로 풀 RAG 파이프라인 (feat. 전자금융감독규정)

## 학습 목표
- 실제 **전자금융감독규정 PDF** 한 건을 끝까지 밀고 가며 01→02→03에서 배운 기술을 **체인 하나로 엮는다**.
- `pymupdf4llm`으로 PDF를 Markdown으로 바꾸고, 조(條) 단위로 청킹한다 (01 리마인드).
- Chroma에 임베딩/색인하고 (02 리마인드), MMR 리트리버를 선택한 이유를 설명한다 (03 리마인드).
- **오늘의 메인**: LCEL(`RunnableParallel` + `.assign()` 누적) 패턴으로 **citation 포함 RAG 체인**을 조립한다.
- 5개 질의에 대한 결과를 `_capstone_baseline.json`으로 저장해 05 노트북과 Day2 S3 캡스톤에서 재사용한다.

## 핵심 키워드
`pymupdf4llm` · `MarkdownHeaderTextSplitter` · `Chroma` · `MMR` · `LCEL` · `RunnableParallel` · `citation` · `capstone baseline`

> 🔒 본 노트북은 **캡스톤 베이스라인**을 구축한다. Day2 Session 3에서 이 마지막 셀의 산출물(`_capstone_baseline.json`)을 이어서 사용한다.


In [ ]:
import sys
sys.path.insert(0, '../..')
from common import get_chat_model, get_embeddings, provider_badge
print(provider_badge())

## 1. PDF → Markdown 변환 (01 리마인드)

01에서 보았듯, PDF는 **구조(제목/조항/목록) 보존**이 검색 품질을 좌우한다. 페이지 loader는 레이아웃 손상이 크므로 여기서는 `pymupdf4llm.to_markdown()`으로 **헤더 계층을 살린 Markdown**을 만든다. 재실행 시 변환을 건너뛰도록 파일 캐시를 둔다.


In [ ]:
from pathlib import Path
import re
import pymupdf4llm

PDF_PATH = Path('../../data/pdf/전자금융감독규정(금융위원회고시)(제2026-7호)(20260213).pdf')
MD_PATH = Path('./_store/efinance_regulation.md')
MD_PATH.parent.mkdir(parents=True, exist_ok=True)

if MD_PATH.exists():
    md_text = MD_PATH.read_text(encoding='utf-8')
    print(f'✅ 캐시 재사용: {MD_PATH.name} ({len(md_text):,} chars)')
else:
    md_text = pymupdf4llm.to_markdown(str(PDF_PATH))
    MD_PATH.write_text(md_text, encoding='utf-8')
    print(f'📄 PDF→MD 변환 완료: {len(md_text):,} chars')

# 페이지 풋터·과도한 공백 정리
md_text = re.sub(r'법제처\s+\d+\s+국가법령정보센터', '', md_text)
md_text = re.sub(r'\n{3,}', '\n\n', md_text)

print('\n— 본문 앞 500자 미리보기 —')
print(md_text[:500])

## 2. 조(條) 단위 청킹 (01 리마인드)

01에서 본 2-stage 전략 — **의미 경계 분할 → 길이 경계 분할** — 을 이 규정에 맞게 적용한다.

- `##` 헤더는 장(章)/절(節)만 잡아주므로 이를 `MarkdownHeaderTextSplitter` 대신 **장 메타데이터**로만 쓴다.
- 실제 답변 단위인 **"제N조(제목)"** 패턴을 정규식으로 잡아 **조(條) 단위 1차 분할**을 수행한다.
- 1,200자를 넘는 긴 조문만 `RecursiveCharacterTextSplitter`로 2차 분할한다. citation을 조 단위로 유지하기 위해 `article` 메타는 동일하게 복제하고 `part` 번호만 다르게 부여한다.


In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

ARTICLE_RE = re.compile(r'(?m)^[\s\-]*제\s*(\d+)\s*조\s*\(([^)]{1,40})\)')
CHAPTER_RE = re.compile(r'(?m)^##\s+(.+?)\s*$')

# 장/절 헤더 위치
chapters = [(m.start(), m.group(1).strip()) for m in CHAPTER_RE.finditer(md_text)]
def chapter_at(pos: int) -> str:
    last = ''
    for s, name in chapters:
        if s <= pos:
            last = name
        else:
            break
    return last

# 조(條) 단위 1차 분할
matches = list(ARTICLE_RE.finditer(md_text))
raw_articles = []
for i, m in enumerate(matches):
    start = m.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(md_text)
    body = md_text[start:end].strip()
    if body.lstrip().startswith('제') is False and body.lstrip().startswith('- 제') is False:
        continue
    if '삭제<' in body[:80]:  # 삭제된 조항 제외
        continue
    raw_articles.append({
        'article': f'제{m.group(1)}조({m.group(2)})',
        'chapter': chapter_at(start),
        'text': body,
    })

# 긴 조문은 2차 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200, chunk_overlap=120,
    separators=['\n\n', '\n', '。', '. ', ' ', ''],
)

chunks = []
for a in raw_articles:
    meta_base = {'source': '전자금융감독규정', 'article': a['article'], 'chapter': a['chapter']}
    if len(a['text']) <= 1200:
        chunks.append(Document(page_content=a['text'], metadata={**meta_base}))
    else:
        for j, piece in enumerate(splitter.split_text(a['text'])):
            chunks.append(Document(page_content=piece, metadata={**meta_base, 'part': j + 1}))

for i, d in enumerate(chunks):
    d.metadata['chunk_id'] = i

avg = sum(len(d.page_content) for d in chunks) / len(chunks)
print(f'조(條): {len(raw_articles)}개 → 최종 chunk: {len(chunks)}개 / 평균 {avg:.0f}자')
print('샘플 메타:', chunks[0].metadata)
print('샘플 본문:', chunks[0].page_content[:100], '…')

## 3. 벡터스토어 (02 리마인드)

02에서 Chroma / FAISS / Qdrant 셋을 비교했다. 이번에는 **로컬 영속화**가 간단하고 Webex 환경에서도 재현이 쉬운 **Chroma**를 선택한다. Qdrant가 더 표현력 있는 필터를 제공하지만, 지금 과제(단일 규정 문서 RAG)에는 과분하다.


In [ ]:
import shutil
from langchain_community.vectorstores import Chroma

CHROMA_DIR = Path('./_store/efinance_rag')
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.parent.mkdir(parents=True, exist_ok=True)

vs = Chroma.from_documents(
    chunks,
    embedding=get_embeddings(),
    persist_directory=str(CHROMA_DIR),
    collection_name='efinance_regulation',
)
print(f'✅ 인덱싱 완료: {vs._collection.count()}개 문서')

## 4. 리트리버 (03 리마인드)

03에서 본 `similarity` / `mmr` / `score_threshold` 중 **MMR**을 택한다. 감독규정은 유사 문구(예: "전자금융기반시설에 대해 … 하여야 한다")가 여러 조에 반복되기 때문에 `similarity`는 한 조에 쏠리기 쉽고, 서로 다른 조를 골고루 가져오는 **다양성 확보**가 근거 인용에 유리하다.


In [ ]:
retriever = vs.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 4, 'fetch_k': 12, 'lambda_mult': 0.5},
)

hits = retriever.invoke('해킹 방지를 위해 요구되는 통제는?')
for d in hits:
    art = d.metadata.get('article', '?')
    chap = d.metadata.get('chapter', '')
    print(f'  [{art}] ({chap[:30]}) {d.page_content[:60]}…')

## 5. LCEL 체인 — 오늘의 메인 이벤트

지금부터는 앞선 리트리버를 **LLM과 한 줄로 묶는다**. LangChain의 LCEL(LangChain Expression Language)은 세 가지 조합 패턴을 기억하면 된다.

1. **`|` 파이프**: 이전 단계의 출력이 다음 단계의 입력으로 흐른다 (`prompt | llm | parser`).
2. **`RunnableParallel`**: 같은 입력을 여러 갈래로 동시에 흘려 dict를 만든다. 여기서 질문과 검색 결과를 **동시에** 확보한다.
3. **`.assign(...)`**: 이미 만든 dict에 **새 키를 추가**한다. 원본 값(예: 검색된 docs)은 버려지지 않고 그대로 들고 간다 → citation UI·감사 로그에 그대로 꽂을 수 있다.

### 5.1 citation 프롬프트
근거가 없으면 **"답변할 수 없습니다"**, 있으면 **`[근거: 제X조]` 목록 출력** — 금융권 컴플라이언스의 기본 원칙이다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

SYSTEM = '''당신은 금융회사의 내부 규정 담당자를 돕는 AI 어시스턴트입니다.
반드시 아래 [컨텍스트]에 근거해 답하세요.

규칙:
1. [컨텍스트]에 근거가 없으면 "제공된 규정에서 기술되지 않아 답변할 수 없습니다." 라고 답합니다.
2. 답변 마지막에 반드시 "[근거: 제X조, 제Y조]" 형식으로 인용한 조항을 모두 나열합니다.
3. 한국어 존댓말, 조항별 핵심 항목만 간결하게 bullet로 정리합니다.'''

USER = '''[질문]
{question}

[컨텍스트]
{context}'''

prompt = ChatPromptTemplate.from_messages([('system', SYSTEM), ('user', USER)])

def format_context(docs):
    lines = []
    for d in docs:
        art = d.metadata.get('article', '알 수 없음')
        lines.append(f'[{art}] {d.page_content}')
    return '\n\n'.join(lines)

### 5.2 체인 조립 — `RunnableParallel` → `.assign()` 누적

아래 한 블록만 읽어도 되는 설계를 목표로 한다.


In [ ]:
llm = get_chat_model(temperature=0)

# 1) RunnableParallel: question은 그대로, retriever는 병렬로 실행해 docs를 만든다.
# 2) .assign(context=...): docs를 프롬프트가 소비 가능한 문자열로 정형화한다.
# 3) .assign(answer=...): 이미 쌓은 키(question/context)를 받아 LLM 답변을 생성한다.
#    이때 docs는 버리지 않고 결과 dict에 남아 있어 UI/로그에서 그대로 쓴다.
rag_chain = (
    RunnableParallel({
        'question': RunnablePassthrough(),
        'docs': retriever,
    })
    .assign(context=lambda x: format_context(x['docs']))
    .assign(answer=prompt | llm | StrOutputParser())
)

print('체인 구성 완료:', type(rag_chain).__name__)

### 5.3 한 건 돌려 보기


In [ ]:
demo_q = '해킹 등 방지대책으로 금융회사가 이행해야 하는 주요 통제항목은 무엇인가?'
out = rag_chain.invoke(demo_q)

print('— Retrieved —')
for d in out['docs']:
    print(f'  [{d.metadata["article"]}] {d.page_content[:70]}…')

print('\n— Answer —')
print(out['answer'])

## 6. 캡스톤 베이스라인 (5개 질의)

아래 5개 질의는 **05_eval_ragas 및 Day2 Session 3 캡스톤**의 입력이 된다. 포맷(`question` / `answer` / `contexts` / `articles`)은 다운스트림 의존성이라 **변경하지 말 것**.


In [ ]:
CAPSTONE_QUESTIONS = [
    '전산실 등 시설부문에 대해 요구되는 물리적·환경적 보안 통제는 무엇인가?',
    '해킹 등 방지대책으로 금융회사가 이행해야 하는 주요 통제항목은?',
    'IP주소 관리대책은 어떤 내용으로 구성되어야 하는가?',
    '전자금융 침해사고에 대비한 비상대응훈련은 어떻게 실시해야 하는가?',
    '정보기술부문의 인력·조직·교육·예산에 대해 규정은 무엇을 요구하는가?',
]

capstone_baseline = []
for q in CAPSTONE_QUESTIONS:
    r = rag_chain.invoke(q)
    capstone_baseline.append({
        'question': q,
        'answer': r['answer'],
        'contexts': [d.page_content for d in r['docs']],
        'articles': [d.metadata.get('article') for d in r['docs']],
    })

for row in capstone_baseline:
    print('Q:', row['question'])
    print('A:', row['answer'])
    print('근거:', row['articles'])
    print('-' * 80)

In [ ]:
import json

out_path = Path('./_capstone_baseline.json')
out_path.write_text(
    json.dumps(capstone_baseline, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'✅ 캡스톤 베이스라인 저장: {out_path.resolve()}')
print(f'   {len(capstone_baseline)}개 질의/답변/컨텍스트 쌍')

## 정리

- **01 리마인드**: PDF는 `pymupdf4llm`으로 Markdown 변환 → 조(條) 단위 1차 분할 → 길이 기반 2차 분할.
- **02 리마인드**: 단일 규정·로컬 영속화·교실 환경 조건에서는 **Chroma**가 가장 실용적.
- **03 리마인드**: 반복 어휘가 많은 규정 검색은 **MMR**로 다양성을 확보.
- **LCEL 3-pattern**: `|` 파이프 · `RunnableParallel` · `.assign()` 누적 — 이 세 개면 대부분의 RAG 체인이 만들어진다.
- **citation + "답변할 수 없습니다"** 패턴은 생성 품질 가드이자 컴플라이언스 근거다.

## 캡스톤 베이스라인

아래 두 산출물이 **다음 노트북(05_eval_ragas)과 Day2 Session 3 캡스톤**에서 그대로 재사용된다.

- `capstone_baseline` — 메모리 상의 리스트 (question / answer / contexts / articles)
- `./_capstone_baseline.json` — 디스크 상의 JSON (다른 노트북에서 로드 가능)

## 더 읽어보기
- [LangChain LCEL Interface](https://python.langchain.com/docs/concepts/lcel/)
- [pymupdf4llm 공식 문서](https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/)
